In [2]:
!pip install holidays -i https://pypi.tuna.tsinghua.edu.cn/simple

import pandas as pd
import numpy as np
import holidays
import warnings
warnings.filterwarnings('ignore')

amazon_path = '/Users/huyujie/Documents/amazon-supply-chain-project/data/raw/amazon_sales_dataset.csv'
uci_path = '/Users/huyujie/Documents/amazon-supply-chain-project/data/raw/Online Retail.xlsx'

df_amazon = pd.read_csv(amazon_path)
df_uci = pd.read_excel(uci_path)

df_amazon['OrderDate'] = pd.to_datetime(df_amazon['OrderDate'])
df_uci['InvoiceDate'] = pd.to_datetime(df_uci['InvoiceDate'])

print("第一块：数据加载与时间转换完成")

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
第一块：数据加载与时间转换完成


In [3]:
df_uci = df_uci[df_uci['UnitPrice'] > 0].copy()

df_amazon = df_amazon.sort_values('OrderDate').reset_index(drop=True)
df_uci = df_uci.sort_values('InvoiceDate').reset_index(drop=True)

amazon_split_idx = int(len(df_amazon) * 0.8)
uci_split_idx = int(len(df_uci) * 0.8)

train_amazon = df_amazon.iloc[:amazon_split_idx].copy()
test_amazon = df_amazon.iloc[amazon_split_idx:].copy()

train_uci = df_uci.iloc[:uci_split_idx].copy()
test_uci = df_uci.iloc[uci_split_idx:].copy()

print("第二块：数据集按时间顺序划分完成")

第二块：数据集按时间顺序划分完成


In [4]:
invoice_to_customer = train_uci.dropna(subset=['CustomerID'])[['InvoiceNo', 'CustomerID']].drop_duplicates()
invoice_to_customer = invoice_to_customer.groupby('InvoiceNo')['CustomerID'].first().to_dict()

def fill_missing_customer(df, mapping_dict):
    df['CustomerID'] = df['CustomerID'].fillna(df['InvoiceNo'].map(mapping_dict))
    df['CustomerID'] = df['CustomerID'].fillna('Unknown_User')
    return df

train_uci = fill_missing_customer(train_uci, invoice_to_customer)
test_uci = fill_missing_customer(test_uci, invoice_to_customer)

print("第三块：训练集CustomerID缺失数量:", train_uci['CustomerID'].isnull().sum())
print("第三块：测试集CustomerID缺失数量:", test_uci['CustomerID'].isnull().sum())

第三块：训练集CustomerID缺失数量: 0
第三块：测试集CustomerID缺失数量: 0


In [5]:
us_holidays = holidays.US()
uk_holidays = holidays.UK()

def add_time_features(df, date_col, country_holidays):
    df['Year'] = df[date_col].dt.year
    df['Month'] = df[date_col].dt.month
    df['Quarter'] = df[date_col].dt.quarter
    df['DayOfWeek'] = df[date_col].dt.dayofweek
    
    df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
    df['IsHoliday'] = df[date_col].dt.date.apply(lambda x: 1 if x in country_holidays else 0)
    df['IsPromotionPeriod'] = df['Month'].apply(lambda x: 1 if x in [11, 12] else 0)
    return df

train_amazon = add_time_features(train_amazon, 'OrderDate', us_holidays)
test_amazon = add_time_features(test_amazon, 'OrderDate', us_holidays)

train_uci = add_time_features(train_uci, 'InvoiceDate', uk_holidays)
test_uci = add_time_features(test_uci, 'InvoiceDate', uk_holidays)

print("第四块：时间序列特征工程构建完成")

第四块：时间序列特征工程构建完成


In [7]:
import os

# 定义最精确的绝对路径文件夹地址，电脑绝对不会迷路
save_dir = '/Users/huyujie/Documents/amazon-supply-chain-project/data/processed/'

# 增加一道保险：让电脑自动检查，如果没找到这个文件夹，就自动帮我们建一个
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"系统发现文件夹不存在，已自动为您创建：{save_dir}")

# 将4个处理好的数据集保存进这个确定的文件夹里
train_amazon.to_csv(save_dir + 'train_amazon_features.csv', index=False)
test_amazon.to_csv(save_dir + 'test_amazon_features.csv', index=False)

train_uci.to_csv(save_dir + 'train_uci_features.csv', index=False)
test_uci.to_csv(save_dir + 'test_uci_features.csv', index=False)

print("第五块：太棒了！所有数据已成功保存至 data/processed/ 目录")

第五块：太棒了！所有数据已成功保存至 data/processed/ 目录
